# DS604 Introduction to Data Management, Autumn 2026
## Milestone 4: Executing Analytical Queries on PostgreSQL
**Group members:** Pratham Patel (202618057), Rohan Deshpande (202618058), Hiteshsinh Rathod (202618040)

**Server:** PostgreSQL at 10.100.71.21

Each query from the Milestone 1 list (Q01 to Q24) is written in SQL, sent to the server with `pandas.read_sql_query`, collected in a DataFrame, and the first 10 rows are printed with `head(10)`.

**Revenue** is defined throughout as `SUM(qty * price)` over `order_items` (as in Milestone 1). `payments.amount` is analysed separately (Q16).

**Data-quality facts from Milestone 2 that affect the results**
- 40,767 orders have no `order_items`, so item-based revenue queries only cover the other 259,233 orders.
- Every order has a promotion, so the "no promotion" group in Q10 is expected to be empty.
- 120,020 orders (40%) are dated before the customer's `signup_date`, so Q24 will show negative day gaps.
- `payments.amount` is unrelated to item totals, so Q16 shows large differences.

## 1. Imports

In [1]:
import getpass
import warnings
from decimal import Decimal
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)
print("pandas", pd.__version__)

C:\Users\Admin\AppData\Roaming\Python\Python312\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\Admin\AppData\Roaming\Python\Python312\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


pandas 3.0.6


## 2. Connect to the PostgreSQL Server


In [2]:
DB_HOST = "10.100.71.21"
DB_PORT = 5432
DB_NAME = input("Database name: ").strip()
DB_USER = input("User name: ").strip()
DB_PASS = getpass.getpass("Password: ")

url = URL.create("postgresql+psycopg2", username=DB_USER, password=DB_PASS,
                 host=DB_HOST, port=DB_PORT, database=DB_NAME)
engine = create_engine(url, pool_pre_ping=True)

with engine.connect() as conn:
    print(conn.execute(text("SELECT version()")).scalar())

PostgreSQL 10.20 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 4.8.5 20150623 (Red Hat 4.8.5-44), 64-bit


## 3. Helper Function


In [3]:
def run_query(sql):
    try:
        with engine.connect() as conn:
            df = pd.read_sql_query(text(sql), conn)
    except Exception as err:
        print("SQLAlchemy path failed (" + type(err).__name__ + "), retrying with raw connection")
        raw = engine.raw_connection()
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                df = pd.read_sql_query(sql, raw)
        finally:
            raw.close()
    for c in df.columns:
        if df[c].dtype == object and len(df) and isinstance(df[c].iloc[0], Decimal):
            df[c] = df[c].astype(float)
    print(f"Rows returned: {len(df):,}")
    return df

## 4. Sanity Check: Row Counts


In [4]:
tables = ["categories","suppliers","stores","promotions","customers","employees",
          "products","orders","order_items","payments","shipments","returns"]
sql = " UNION ALL ".join(f"SELECT '{t}' AS table_name, COUNT(*) AS n_rows FROM {t}" for t in tables)
df = run_query(sql)
df.head(10)

Rows returned: 12


,table_name,n_rows
0,categories,30
1,suppliers,200
2,stores,100
3,promotions,50
4,customers,50000
5,employees,1000
6,products,10000
7,orders,300000
8,order_items,600000
9,payments,300000


## 5. Analytical Queries (Q01 to Q24)

### Q01. Customers by city
**Question:** Count customers in each city and show the percentage of total customers.

**SQL concepts:** GROUP BY, COUNT, window function for percentage

In [5]:
sql_q01 = """
SELECT city,
       COUNT(*) AS customers,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_of_total
FROM customers
GROUP BY city
ORDER BY customers DESC;
"""
df_q01 = run_query(sql_q01)
df_q01.head(10)

Rows returned: 4


,city,customers,pct_of_total
0,Mumbai,12656,25.31
1,Pune,12521,25.04
2,Bangalore,12434,24.87
3,Delhi,12389,24.78


### Q02. Customer sign-up trend
**Question:** Count new customers per year and month using signup_date.

**SQL concepts:** Date functions, GROUP BY

In [6]:
sql_q02 = """
SELECT EXTRACT(YEAR FROM signup_date)  AS signup_year,
       EXTRACT(MONTH FROM signup_date) AS signup_month,
       COUNT(*) AS new_customers
FROM customers
GROUP BY 1, 2
ORDER BY 1, 2;
"""
df_q02 = run_query(sql_q02)
df_q02.head(10)

Rows returned: 61


,signup_year,signup_month,new_customers
0,2019.0,1.0,834
1,2019.0,2.0,729
2,2019.0,3.0,877
3,2019.0,4.0,780
4,2019.0,5.0,833
5,2019.0,6.0,812
6,2019.0,7.0,922
7,2019.0,8.0,828
8,2019.0,9.0,791
9,2019.0,10.0,878


### Q03. Sales by store
**Question:** For each store (and its city) calculate the number of orders and total revenue, ordered by revenue.

**SQL concepts:** JOIN, SUM, GROUP BY, ORDER BY

**Note:** Only orders that have at least one line item are counted.

In [7]:
sql_q03 = """
SELECT s.store_id, s.city,
       COUNT(DISTINCT o.order_id) AS orders,
       SUM(oi.qty * oi.price)     AS revenue
FROM stores s
JOIN orders o       ON o.store_id = s.store_id
JOIN order_items oi ON oi.order_id = o.order_id
GROUP BY s.store_id, s.city
ORDER BY revenue DESC;
"""
df_q03 = run_query(sql_q03)
df_q03.head(10)

Rows returned: 100


,store_id,city,orders,revenue
0,9,Delhi,2671,40093705.0
1,72,Pune,2698,40082772.0
2,48,Mumbai,2672,40030373.0
3,4,Mumbai,2660,39929457.0
4,67,Pune,2668,39851073.0
5,55,Mumbai,2648,39849329.0
6,7,Delhi,2727,39833700.0
7,54,Mumbai,2651,39830349.0
8,8,Bangalore,2703,39794678.0
9,34,Mumbai,2655,39635295.0


### Q04. Top products by revenue
**Question:** Find the top 10 products by total revenue and units sold.

**SQL concepts:** JOIN, SUM, GROUP BY, LIMIT

In [8]:
sql_q04 = """
SELECT p.product_id,
       SUM(oi.qty)            AS units_sold,
       SUM(oi.qty * oi.price) AS revenue
FROM order_items oi
JOIN products p ON p.product_id = oi.product_id
GROUP BY p.product_id
ORDER BY revenue DESC
LIMIT 10;
"""
df_q04 = run_query(sql_q04)
df_q04.head(10)

Rows returned: 10


,product_id,units_sold,revenue
0,745,236,674395.0
1,8248,237,653199.0
2,4738,196,633367.0
3,7363,219,624564.0
4,9133,216,620725.0
5,5485,217,619073.0
6,1216,190,614960.0
7,5201,218,610779.0
8,6302,219,600997.0
9,1584,220,598328.0


### Q05. Category performance
**Question:** Compare units sold, revenue and number of orders across product categories.

**SQL concepts:** Multiple JOINs, GROUP BY, multiple aggregates

In [9]:
sql_q05 = """
SELECT c.category_name,
       SUM(oi.qty)                  AS units_sold,
       SUM(oi.qty * oi.price)       AS revenue,
       COUNT(DISTINCT oi.order_id)  AS orders
FROM order_items oi
JOIN products p   ON p.product_id = oi.product_id
JOIN categories c ON c.category_id = p.category_id
GROUP BY c.category_name
ORDER BY revenue DESC;
"""
df_q05 = run_query(sql_q05)
df_q05.head(10)

Rows returned: 30


,category_name,units_sold,revenue,orders
0,Cat_5,53906,136704132.0,20661
1,Cat_3,52985,135379339.0,20463
2,Cat_21,52674,135329061.0,20367
3,Cat_26,52526,134697069.0,20397
4,Cat_12,52804,133897588.0,20378
5,Cat_28,52329,133603891.0,20254
6,Cat_8,52215,132816590.0,20232
7,Cat_9,52270,132465085.0,20221
8,Cat_13,51867,132409449.0,20015
9,Cat_17,51886,132292751.0,20039


### Q06. Supplier revenue by country
**Question:** Find which supplier countries contribute the most revenue through their products.

**SQL concepts:** Multiple JOINs, SUM, GROUP BY

In [10]:
sql_q06 = """
SELECT su.country,
       COUNT(DISTINCT su.supplier_id) AS suppliers,
       SUM(oi.qty * oi.price)         AS revenue
FROM order_items oi
JOIN products p   ON p.product_id = oi.product_id
JOIN suppliers su ON su.supplier_id = p.supplier_id
GROUP BY su.country
ORDER BY revenue DESC;
"""
df_q06 = run_query(sql_q06)
df_q06.head(10)

Rows returned: 3


,country,suppliers,revenue
0,USA,75,1.442391e+09
1,China,64,1.212465e+09
2,India,61,1.172891e+09


### Q07. Average order value by store city
**Question:** Calculate total revenue per order first, then the average order value for each store city.

**SQL concepts:** CTE, AVG, JOIN

In [11]:
sql_q07 = """
WITH order_total AS (
    SELECT order_id, SUM(qty * price) AS total
    FROM order_items
    GROUP BY order_id
)
SELECT s.city,
       COUNT(*)                  AS orders,
       ROUND(AVG(ot.total), 2)   AS avg_order_value
FROM orders o
JOIN order_total ot ON ot.order_id = o.order_id
JOIN stores s       ON s.store_id = o.store_id
GROUP BY s.city
ORDER BY avg_order_value DESC;
"""
df_q07 = run_query(sql_q07)
df_q07.head(10)

Rows returned: 4


,city,orders,avg_order_value
0,Mumbai,80379,14804.79
1,Delhi,54446,14798.58
2,Pune,70240,14745.80
3,Bangalore,54168,14700.25


### Q08. Monthly revenue trend
**Question:** Calculate monthly orders, units sold and revenue across the data period.

**SQL concepts:** DATE_TRUNC, GROUP BY

In [12]:
sql_q08 = """
SELECT CAST(DATE_TRUNC('month', o.order_date) AS DATE) AS month,
       COUNT(DISTINCT o.order_id) AS orders,
       SUM(oi.qty)                AS units_sold,
       SUM(oi.qty * oi.price)     AS revenue
FROM orders o
JOIN order_items oi ON oi.order_id = o.order_id
GROUP BY 1
ORDER BY 1;
"""
df_q08 = run_query(sql_q08)
df_q08.head(10)

Rows returned: 49


,month,orders,units_sold,revenue
0,2020-01-01,5560,32142,82204655.0
1,2020-02-01,5199,29523,75794816.0
2,2020-03-01,5545,32195,82462951.0
3,2020-04-01,5370,30710,78006033.0
4,2020-05-01,5534,31968,81583526.0
5,2020-06-01,5325,30430,77776543.0
6,2020-07-01,5570,32375,82868115.0
7,2020-08-01,5526,31875,80465611.0
8,2020-09-01,5217,30336,77113009.0
9,2020-10-01,5512,31470,79518804.0


### Q09. Year-over-year revenue growth
**Question:** Calculate yearly revenue and the percentage change from the previous year.

**SQL concepts:** CTE, LAG window function

**Note:** The first year has no previous year, so its growth is NULL (NaN).

In [13]:
sql_q09 = """
WITH yearly AS (
    SELECT EXTRACT(YEAR FROM o.order_date) AS year,
           SUM(oi.qty * oi.price)          AS revenue
    FROM orders o
    JOIN order_items oi ON oi.order_id = o.order_id
    GROUP BY 1
)
SELECT year,
       revenue,
       LAG(revenue) OVER (ORDER BY year) AS prev_year_revenue,
       ROUND(100.0 * (revenue - LAG(revenue) OVER (ORDER BY year))
             / LAG(revenue) OVER (ORDER BY year), 2) AS yoy_growth_pct
FROM yearly
ORDER BY year;
"""
df_q09 = run_query(sql_q09)
df_q09.head(10)

Rows returned: 5


,year,revenue,prev_year_revenue,yoy_growth_pct
0,2020.0,959634059.0,NaN,NaN
1,2021.0,950027068.0,959634059.0,-1.00
2,2022.0,957707647.0,950027068.0,0.81
3,2023.0,957899902.0,957707647.0,0.02
4,2024.0,2477460.0,957899902.0,-99.74


### Q10. Promotion vs non-promotion orders
**Question:** Compare order count, revenue and average order value for orders with and without a promotion.

**SQL concepts:** LEFT JOIN, CASE, GROUP BY

**Note:** Every order in this dataset has a promotion, so only the 'With promotion' group appears. The query still works if NULL promotions are added later.

In [14]:
sql_q10 = """
WITH order_total AS (
    SELECT order_id, SUM(qty * price) AS total
    FROM order_items
    GROUP BY order_id
)
SELECT CASE WHEN p.promotion_id IS NULL THEN 'No promotion' ELSE 'With promotion' END AS promo_group,
       COUNT(*)                AS orders,
       SUM(ot.total)           AS revenue,
       ROUND(AVG(ot.total), 2) AS avg_order_value
FROM orders o
JOIN order_total ot    ON ot.order_id = o.order_id
LEFT JOIN promotions p ON p.promotion_id = o.promotion_id
GROUP BY 1
ORDER BY 1;
"""
df_q10 = run_query(sql_q10)
df_q10.head(10)

Rows returned: 1


,promo_group,orders,revenue,avg_order_value
0,With promotion,259233,3.827746e+09,14765.66


### Q11. Revenue by discount level
**Question:** Group orders by promotion discount value and compare order volume and revenue.

**SQL concepts:** JOIN, GROUP BY, AVG, SUM

In [15]:
sql_q11 = """
WITH order_total AS (
    SELECT order_id, SUM(qty * price) AS total
    FROM order_items
    GROUP BY order_id
)
SELECT p.discount,
       COUNT(*)                AS orders,
       SUM(ot.total)           AS revenue,
       ROUND(AVG(ot.total), 2) AS avg_order_value
FROM orders o
JOIN order_total ot ON ot.order_id = o.order_id
JOIN promotions p   ON p.promotion_id = o.promotion_id
GROUP BY p.discount
ORDER BY p.discount;
"""
df_q11 = run_query(sql_q11)
df_q11.head(10)

Rows returned: 27


,discount,orders,revenue,avg_order_value
0,5,20641,304797932.0,14766.63
1,6,20772,304976095.0,14682.08
2,7,10365,153416444.0,14801.39
3,9,10250,150415116.0,14674.65
4,10,5358,77925743.0,14543.81
5,11,10212,152408425.0,14924.44
6,13,5271,76902495.0,14589.74
7,14,5026,73498548.0,14623.67
8,15,10271,149392466.0,14545.08
9,16,10442,151813566.0,14538.74


### Q12. Return rate by category
**Question:** Calculate the share of order lines that were returned for each product category.

**SQL concepts:** JOIN, COUNT, LEFT JOIN, conditional counting

**Note:** 731 order lines have more than one return row, so returns are de-duplicated per line before counting.

In [16]:
sql_q12 = """
WITH returned AS (
    SELECT DISTINCT order_item_id FROM returns
)
SELECT c.category_name,
       COUNT(*)                  AS order_lines,
       COUNT(rt.order_item_id)   AS returned_lines,
       ROUND(100.0 * COUNT(rt.order_item_id) / COUNT(*), 2) AS return_rate_pct
FROM order_items oi
JOIN products p   ON p.product_id = oi.product_id
JOIN categories c ON c.category_id = p.category_id
LEFT JOIN returned rt ON rt.order_item_id = oi.order_item_id
GROUP BY c.category_name
ORDER BY return_rate_pct DESC;
"""
df_q12 = run_query(sql_q12)
df_q12.head(10)

Rows returned: 30


,category_name,order_lines,returned_lines,return_rate_pct
0,Cat_11,20300,1068,5.26
1,Cat_14,18689,958,5.13
2,Cat_26,21129,1070,5.06
3,Cat_16,20618,1037,5.03
4,Cat_21,21066,1058,5.02
5,Cat_24,18930,951,5.02
6,Cat_7,18982,949,5.00
7,Cat_8,20950,1045,4.99
8,Cat_19,19499,972,4.98
9,Cat_10,20312,1011,4.98


### Q13. Refunds by supplier country
**Question:** Find total refund value and number of returns linked to each supplier country.

**SQL concepts:** Multi-table JOIN, SUM, COUNT

In [17]:
sql_q13 = """
SELECT su.country,
       COUNT(*)        AS returns,
       SUM(r.refund)   AS total_refund
FROM returns r
JOIN order_items oi ON oi.order_item_id = r.order_item_id
JOIN products p     ON p.product_id = oi.product_id
JOIN suppliers su   ON su.supplier_id = p.supplier_id
GROUP BY su.country
ORDER BY total_refund DESC;
"""
df_q13 = run_query(sql_q13)
df_q13.head(10)

Rows returned: 3


,country,returns,total_refund
0,USA,11239,28426162.0
1,China,9547,24117657.0
2,India,9214,23418481.0


### Q14. Shipment status distribution
**Question:** Count shipments by status and show each status as a percentage of all shipments.

**SQL concepts:** GROUP BY, COUNT, window percentage

In [18]:
sql_q14 = """
SELECT status,
       COUNT(*) AS shipments,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_of_total
FROM shipments
GROUP BY status
ORDER BY shipments DESC;
"""
df_q14 = run_query(sql_q14)
df_q14.head(10)

Rows returned: 3


,status,shipments,pct_of_total
0,shipped,100283,33.43
1,delivered,99862,33.29
2,late,99855,33.29


### Q15. Orders with missing shipment or payment
**Question:** Find orders that have no shipment record, or no payment record.

**SQL concepts:** LEFT JOIN, IS NULL (anti-join)

**Note:** Milestone 2 found exactly one payment and one shipment per order, so this is expected to return 0 rows (a clean result). The summary below confirms it.

In [19]:
sql_q15 = """
SELECT o.order_id, o.order_date,
       CASE WHEN s.shipment_id IS NULL THEN 'missing' ELSE 'ok' END AS shipment,
       CASE WHEN p.payment_id  IS NULL THEN 'missing' ELSE 'ok' END AS payment
FROM orders o
LEFT JOIN shipments s ON s.order_id = o.order_id
LEFT JOIN payments  p ON p.order_id = o.order_id
WHERE s.shipment_id IS NULL OR p.payment_id IS NULL
ORDER BY o.order_id;
"""
df_q15 = run_query(sql_q15)
df_q15.head(10)

Rows returned: 0


,order_id,order_date,shipment,payment


### Q15b. Q15 summary counts
**Question:** Summary of order-lifecycle completeness.

**SQL concepts:** Conditional aggregation

In [20]:
sql_q15b = """
SELECT COUNT(*) AS total_orders,
       COUNT(*) FILTER (WHERE NOT EXISTS (SELECT 1 FROM shipments s WHERE s.order_id = o.order_id)) AS no_shipment,
       COUNT(*) FILTER (WHERE NOT EXISTS (SELECT 1 FROM payments p  WHERE p.order_id = o.order_id)) AS no_payment,
       COUNT(*) FILTER (WHERE NOT EXISTS (SELECT 1 FROM order_items i WHERE i.order_id = o.order_id)) AS no_items
FROM orders o;
"""
df_q15b = run_query(sql_q15b)
df_q15b.head(10)

Rows returned: 1


,total_orders,no_shipment,no_payment,no_items
0,300000,0,0,40767


### Q16. Payment vs order total
**Question:** Compare payments.amount with the line-item total of each order and list the largest differences.

**SQL concepts:** CTE, JOIN, ABS, ORDER BY

**Note:** Milestone 2 found that only 10 of 259,233 orders have a payment equal to the item total, so large differences are expected. This is why revenue is defined from order_items.

In [21]:
sql_q16 = """
WITH item_total AS (
    SELECT order_id, SUM(qty * price) AS items_total
    FROM order_items
    GROUP BY order_id
), paid AS (
    SELECT order_id, SUM(amount) AS paid_total
    FROM payments
    GROUP BY order_id
)
SELECT it.order_id, it.items_total, pd.paid_total,
       ABS(pd.paid_total - it.items_total) AS abs_difference
FROM item_total it
JOIN paid pd ON pd.order_id = it.order_id
ORDER BY abs_difference DESC
LIMIT 100;
"""
df_q16 = run_query(sql_q16)
df_q16.head(10)

Rows returned: 100


,order_id,items_total,paid_total,abs_difference
0,217950,94157.0,2553.0,91604.0
1,176223,99276.0,10566.0,88710.0
2,249625,83790.0,2383.0,81407.0
3,276700,84479.0,3372.0,81107.0
4,100344,83731.0,5412.0,78319.0
5,140934,79316.0,1132.0,78184.0
6,123028,88180.0,11737.0,76443.0
7,99222,86887.0,10561.0,76326.0
8,37498,75468.0,572.0,74896.0
9,118707,74999.0,692.0,74307.0


### Q17. Repeat customers
**Question:** Identify customers with more than one order and calculate the repeat-customer rate.

**SQL concepts:** COUNT, GROUP BY, HAVING

In [22]:
sql_q17 = """
SELECT customer_id, COUNT(*) AS orders
FROM orders
GROUP BY customer_id
HAVING COUNT(*) > 1
ORDER BY orders DESC, customer_id
LIMIT 100;
"""
df_q17 = run_query(sql_q17)
df_q17.head(10)

Rows returned: 100


,customer_id,orders
0,31428,19
1,518,17
2,2031,17
3,14913,17
4,31761,17
5,33455,17
6,8141,16
7,9140,16
8,15151,16
9,15820,16


### Q17b. Repeat-customer rate
**Question:** Share of ordering customers who placed more than one order.

**SQL concepts:** CTE, conditional counting

In [23]:
sql_q17b = """
WITH per_customer AS (
    SELECT customer_id, COUNT(*) AS n
    FROM orders
    GROUP BY customer_id
)
SELECT COUNT(*) AS customers_with_orders,
       COUNT(*) FILTER (WHERE n > 1) AS repeat_customers,
       ROUND(100.0 * COUNT(*) FILTER (WHERE n > 1) / COUNT(*), 2) AS repeat_rate_pct
FROM per_customer;
"""
df_q17b = run_query(sql_q17b)
df_q17b.head(10)

Rows returned: 1


,customers_with_orders,repeat_customers,repeat_rate_pct
0,49886,49168,98.56


### Q18. Customer lifetime value
**Question:** Find the top 10 customers by total spend and show their order counts.

**SQL concepts:** JOIN, SUM, GROUP BY, LIMIT

In [24]:
sql_q18 = """
SELECT o.customer_id,
       COUNT(DISTINCT o.order_id) AS orders,
       SUM(oi.qty * oi.price)     AS total_spend
FROM orders o
JOIN order_items oi ON oi.order_id = o.order_id
GROUP BY o.customer_id
ORDER BY total_spend DESC
LIMIT 10;
"""
df_q18 = run_query(sql_q18)
df_q18.head(10)

Rows returned: 10


,customer_id,orders,total_spend
0,33455,15,316629.0
1,9140,15,295993.0
2,46109,14,290435.0
3,47418,10,287538.0
4,30723,10,284477.0
5,13511,13,284287.0
6,5101,13,283708.0
7,42816,14,283484.0
8,13527,13,282044.0
9,8141,15,281759.0


### Q19. Customers who never ordered
**Question:** List customers with no orders and summarise them by city and sign-up year.

**SQL concepts:** LEFT JOIN, IS NULL, GROUP BY

**Note:** If no customers qualify, the result is empty (all 50,000 customers placed at least one order).

In [25]:
sql_q19 = """
SELECT c.city,
       EXTRACT(YEAR FROM c.signup_date) AS signup_year,
       COUNT(*) AS customers_without_orders
FROM customers c
LEFT JOIN orders o ON o.customer_id = c.customer_id
WHERE o.order_id IS NULL
GROUP BY c.city, EXTRACT(YEAR FROM c.signup_date)
ORDER BY c.city, signup_year;
"""
df_q19 = run_query(sql_q19)
df_q19.head(10)

Rows returned: 20


,city,signup_year,customers_without_orders
0,Bangalore,2019.0,4
1,Bangalore,2020.0,4
2,Bangalore,2021.0,6
3,Bangalore,2022.0,7
4,Bangalore,2023.0,7
5,Delhi,2019.0,6
6,Delhi,2020.0,5
7,Delhi,2021.0,5
8,Delhi,2022.0,3
9,Delhi,2023.0,5


### Q20. Top customers within each city
**Question:** Rank customers by total spend within each city and keep the top 3 per city.

**SQL concepts:** Window function, PARTITION BY, RANK

In [26]:
sql_q20 = """
WITH spend AS (
    SELECT c.customer_id, c.city, SUM(oi.qty * oi.price) AS total_spend
    FROM customers c
    JOIN orders o       ON o.customer_id = c.customer_id
    JOIN order_items oi ON oi.order_id = o.order_id
    GROUP BY c.customer_id, c.city
), ranked AS (
    SELECT *, RANK() OVER (PARTITION BY city ORDER BY total_spend DESC) AS rnk
    FROM spend
)
SELECT city, rnk, customer_id, total_spend
FROM ranked
WHERE rnk <= 3
ORDER BY city, rnk;
"""
df_q20 = run_query(sql_q20)
df_q20.head(10)

Rows returned: 12


,city,rnk,customer_id,total_spend
0,Bangalore,1,9140,295993.0
1,Bangalore,2,47418,287538.0
2,Bangalore,3,5101,283708.0
3,Delhi,1,33455,316629.0
4,Delhi,2,46109,290435.0
5,Delhi,3,30723,284477.0
6,Mumbai,1,20912,280589.0
7,Mumbai,2,27580,275256.0
8,Mumbai,3,16262,271202.0
9,Pune,1,49980,277486.0


### Q21. Top products within each category
**Question:** Rank products within each category by revenue and keep the top 3 per category.

**SQL concepts:** DENSE_RANK, PARTITION BY

In [27]:
sql_q21 = """
WITH prod_rev AS (
    SELECT p.category_id, p.product_id, SUM(oi.qty * oi.price) AS revenue
    FROM order_items oi
    JOIN products p ON p.product_id = oi.product_id
    GROUP BY p.category_id, p.product_id
), ranked AS (
    SELECT *, DENSE_RANK() OVER (PARTITION BY category_id ORDER BY revenue DESC) AS rnk
    FROM prod_rev
)
SELECT c.category_name, r.rnk, r.product_id, r.revenue
FROM ranked r
JOIN categories c ON c.category_id = r.category_id
WHERE r.rnk <= 3
ORDER BY c.category_name, r.rnk;
"""
df_q21 = run_query(sql_q21)
df_q21.head(10)

Rows returned: 90


,category_name,rnk,product_id,revenue
0,Cat_1,1,1868,566880.0
1,Cat_1,2,631,531139.0
2,Cat_1,3,3188,527366.0
3,Cat_10,1,6302,600997.0
4,Cat_10,2,804,549277.0
5,Cat_10,3,2537,546136.0
6,Cat_11,1,6129,538196.0
7,Cat_11,2,2563,529376.0
8,Cat_11,3,4011,529219.0
9,Cat_12,1,1584,598328.0


### Q22. Employees above store average salary
**Question:** List employees earning more than the average salary of their own store.

**SQL concepts:** Subquery/CTE, AVG, JOIN

In [28]:
sql_q22 = """
WITH store_avg AS (
    SELECT store_id, AVG(salary) AS avg_salary
    FROM employees
    GROUP BY store_id
)
SELECT e.employee_id, e.store_id, e.salary,
       ROUND(sa.avg_salary, 2) AS store_avg_salary
FROM employees e
JOIN store_avg sa ON sa.store_id = e.store_id
WHERE e.salary > sa.avg_salary
ORDER BY e.store_id, e.salary DESC;
"""
df_q22 = run_query(sql_q22)
df_q22.head(10)

Rows returned: 494


,employee_id,store_id,salary,store_avg_salary
0,21,1,67882.0,45535.20
1,326,1,63129.0,45535.20
2,409,1,56042.0,45535.20
3,636,1,52415.0,45535.20
4,717,1,50766.0,45535.20
5,169,1,46698.0,45535.20
6,423,2,74500.0,53118.45
7,302,2,66132.0,53118.45
8,841,2,65803.0,53118.45
9,111,2,61444.0,53118.45


### Q23. Revenue per employee by store
**Question:** Compare each store's revenue to its employee count and salary cost.

**SQL concepts:** CTE, multiple aggregates, JOIN

**Note:** Revenue and employees are aggregated in separate CTEs first, so the joins do not inflate the totals. Stores with no employees are excluded by the inner join.

In [29]:
sql_q23 = """
WITH store_rev AS (
    SELECT o.store_id, SUM(oi.qty * oi.price) AS revenue
    FROM orders o
    JOIN order_items oi ON oi.order_id = o.order_id
    GROUP BY o.store_id
), store_emp AS (
    SELECT store_id, COUNT(*) AS employees, SUM(salary) AS salary_cost
    FROM employees
    GROUP BY store_id
)
SELECT s.store_id, s.city, sr.revenue, se.employees, se.salary_cost,
       ROUND(sr.revenue / se.employees, 2) AS revenue_per_employee
FROM stores s
JOIN store_rev sr ON sr.store_id = s.store_id
JOIN store_emp se ON se.store_id = s.store_id
ORDER BY revenue_per_employee DESC;
"""
df_q23 = run_query(sql_q23)
df_q23.head(10)

Rows returned: 100


,store_id,city,revenue,employees,salary_cost,revenue_per_employee
0,96,Pune,37464060.0,4,183142.0,9366015.00
1,87,Mumbai,36176421.0,4,196645.0,9044105.25
2,65,Pune,39237602.0,5,235081.0,7847520.40
3,38,Mumbai,39166354.0,5,271446.0,7833270.80
4,28,Mumbai,38658232.0,5,266897.0,7731646.40
5,46,Delhi,38129689.0,5,288948.0,7625937.80
6,98,Mumbai,37854618.0,5,246953.0,7570923.60
7,10,Bangalore,36983109.0,5,302890.0,7396621.80
8,69,Delhi,39154951.0,6,279761.0,6525825.17
9,78,Pune,38458253.0,6,282878.0,6409708.83


### Q24. Time from sign-up to first order
**Question:** For each customer calculate the days between signup_date and the first order, then summarise the average by city.

**SQL concepts:** MIN, date arithmetic, GROUP BY

**Note:** Milestone 2 found that 40% of orders are dated before the customer's sign-up date, so negative gaps are a data-quality issue in the source data, not a query error.

In [30]:
sql_q24 = """
WITH first_order AS (
    SELECT customer_id, MIN(order_date) AS first_order_date
    FROM orders
    GROUP BY customer_id
), gap AS (
    SELECT c.customer_id, c.city, (f.first_order_date - c.signup_date) AS days_to_first_order
    FROM customers c
    JOIN first_order f ON f.customer_id = c.customer_id
)
SELECT city,
       COUNT(*)                                          AS customers,
       ROUND(AVG(days_to_first_order), 1)                AS avg_days,
       MIN(days_to_first_order)                          AS min_days,
       MAX(days_to_first_order)                          AS max_days,
       COUNT(*) FILTER (WHERE days_to_first_order < 0)   AS first_order_before_signup
FROM gap
GROUP BY city
ORDER BY avg_days;
"""
df_q24 = run_query(sql_q24)
df_q24.head(10)

Rows returned: 4


,city,customers,avg_days,min_days,max_days,first_order_before_signup
0,Mumbai,12627,-316.1,-1453,1636,8467
1,Delhi,12365,-314.7,-1456,1617,8311
2,Bangalore,12406,-309.3,-1460,1705,8333
3,Pune,12488,-306.0,-1450,1713,8277


## 6. Close the Connection

In [31]:
engine.dispose()
print("Connection closed.")

Connection closed.


## 7. Summary
All 24 analytical queries from Milestone 1 (plus two small helper queries, Q15b and Q17b) were run on the PostgreSQL server through `pandas.read_sql_query`. Each result was collected in a DataFrame and its first 10 rows printed with `head(10)`.

The queries use JOINs across up to four tables, GROUP BY with HAVING, CTEs, window functions (RANK, DENSE_RANK, LAG, SUM OVER), anti-joins, subqueries, conditional aggregation and date arithmetic.

Where the Milestone 2 data-quality findings matter (orders without items, payments unrelated to item totals, orders before sign-up, all orders having a promotion), the notes under each query explain how the result should be read.